In [1]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import tiktoken

In [2]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

In [3]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


In [4]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))))

In [5]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"])
        )

    def forward(self, x):
        return self.layers(x)

In [6]:
class MaskedMultiheadAttention(nn.Module):
    def __init__(self, context_length, dim_in, dim_out, n_heads, dropout_rate, qkv_bias=False):
        super().__init__()
        
        self.context_length = context_length
        self.dim_in = dim_in
        self.dim_out = dim_out
        self.n_heads = n_heads

        assert self.dim_out % self.n_heads == 0, "dim_out should be divisible by n_heads"
        
        self.head_dim = self.dim_out // self.n_heads
        self.W_key = nn.Linear(self.dim_in, self.dim_out, bias=qkv_bias)
        self.W_query = nn.Linear(self.dim_in, self.dim_out, bias=qkv_bias)
        self.W_value = nn.Linear(self.dim_in, self.dim_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout_rate)
        self.out_proj = nn.Linear(self.dim_out, self.dim_out)
        self.register_buffer("mask", torch.triu(torch.ones(self.context_length, self.context_length), diagonal=1).bool())

    def forward(self, x):
        batch_size, num_tokens, dim_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(batch_size, num_tokens, self.n_heads, self.head_dim)
        queries = queries.view(batch_size, num_tokens, self.n_heads, self.head_dim)
        values = values.view(batch_size, num_tokens, self.n_heads, self.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attention_scores = queries @ keys.transpose(2, 3)
        attention_scores.masked_fill_(self.mask[:num_tokens, :num_tokens], -torch.inf)
        attention_weights = torch.softmax(attention_scores / (keys.shape[-1] ** 0.5), dim=-1)
        attention_weights = self.dropout(attention_weights)
        context_vectors = (attention_weights @ values).transpose(1, 2)
        context_vectors = context_vectors.contiguous().view(batch_size, num_tokens, self.dim_out)
        context_vectors = self.out_proj(context_vectors)
        return context_vectors

In [7]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attention = MaskedMultiheadAttention(
            context_length = cfg["context_length"],
            dim_in = cfg["emb_dim"],
            dim_out = cfg["emb_dim"],
            dropout_rate = cfg["drop_rate"],
            qkv_bias = cfg["qkv_bias"],
            n_heads = cfg["n_heads"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.attention(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x

In [8]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_embedding = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_embedding = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_embedding(in_idx)
        pos_embeds = self.pos_embedding(torch.arange(seq_len, device=in_idx.device))

        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [10]:
def total_params(model):
    return sum(p.numel() for p in model.parameters())

In [14]:
torch.manual_seed(123)
batch = []
tx1 = "Every effort moves you"
tx2 = "Every day holds a"
tokenizer = tiktoken.get_encoding("gpt2")

batch.append(torch.tensor(tokenizer.encode(tx1)))
batch.append(torch.tensor(tokenizer.encode(tx2)))
batch = torch.stack(batch, dim=0)
model = GPTModel(GPT_CONFIG_124M)
model(batch)

tensor([[[ 0.3613,  0.4222, -0.0711,  ...,  0.3483,  0.4661, -0.2838],
         [-0.1527, -0.5039, -0.8502,  ...,  0.0877,  0.5718, -0.3429],
         [ 0.7497,  0.0502,  0.0263,  ...,  0.0523, -0.4993, -0.1765],
         [-0.9096,  0.4484, -0.1124,  ...,  0.7928,  0.4427, -0.0017]],

        [[-0.2564,  0.0900,  0.0335,  ...,  0.2659,  0.4454, -0.6806],
         [ 0.1317,  0.4324, -0.1969,  ...,  0.8463,  0.2101,  0.1708],
         [ 1.0335,  1.0049, -0.2195,  ...,  0.6317,  0.3796, -0.2901],
         [-0.1305,  0.3932,  0.3877,  ...,  1.2653, -0.1867, -0.0026]]],
       grad_fn=<UnsafeViewBackward0>)

In [15]:
model = GPTModel(GPT_CONFIG_124M)

In [16]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
            logits = logits[:, -1, :]
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.argmax(probs, dim=-1, keepdim=True)
            idx = torch.cat((idx, idx_next), dim=1)

    return idx

In [35]:
start_context = "a for "
encoded = tokenizer.encode(start_context)
print("encoded ", encoded)
encoded_tensor = torch.tensor(encoded).unsqueeze(0)
print("encoded tensor shape", encoded_tensor.shape)

encoded  [64, 329, 220]
encoded tensor shape torch.Size([1, 3])


In [36]:
model.eval()
out = generate_text_simple(model, idx=encoded_tensor, max_new_tokens=6, context_size=GPT_CONFIG_124M["context_length"])

In [37]:
print(out, out.shape)
tokenizer.decode(out.squeeze(0).tolist())

tensor([[   64,   329,   220, 17687, 23677, 45919, 20056, 21830, 43668]]) torch.Size([1, 9])


'a for  reversed translates Laos summoned Comcast Moderate'

In [41]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256, # use a shorter context length
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

In [39]:
model = GPTModel(GPT_CONFIG_124M)

In [40]:
total_params(model)

163009536

In [45]:
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())

In [48]:
start_context = "Every effort moves you to hell"

model.eval()
out = generate_text_simple(model, idx=text_to_token_ids(start_context, tokenizer), max_new_tokens=6, context_size=GPT_CONFIG_124M["context_length"])
print("Output \n", token_ids_to_text(out, tokenizer))

Output 
 Every effort moves you to hell creamy Ut Repe Wink climbing Fant
